# 🧠 Step 4: Memory Patterns

## Making the System Learn

Our workflow handles incidents — but every time it starts from scratch.
A human SRE gets better over time because they **remember** past incidents.

In this step, we'll add memory so our agents can:
- Store resolutions from past incidents
- Retrieve relevant context when similar issues occur
- Build institutional knowledge that persists

## Key Concept: Context Providers

MAF provides `ContextProvider` — a way to inject relevant context into an agent's
prompt at runtime. The agent doesn't need to "remember" everything; it gets the
relevant memories injected automatically.

```
Alert fires → Memory Provider retrieves similar past incidents
            → Injects resolution context into Triage Agent prompt
            → Agent says: "This is the same as INC-12345, fixed by restarting pod-3"
```

In [ ]:
import os
import json
from datetime import datetime
from dataclasses import dataclass, field
from dotenv import load_dotenv

from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv()
print("✅ Imports ready")

## Build an Incident Memory Store

We'll create a simple in-memory store that holds past incident resolutions.
In production, this would be backed by Azure AI Search, CosmosDB, or Redis.

### 🎯 YOUR TASK
Implement the `search` method to find relevant past incidents based on the
service name and keywords.

In [ ]:
@dataclass
class IncidentMemory:
    """Simple incident resolution memory store."""
    service: str
    title: str
    root_cause: str
    resolution: str
    timestamp: str
    time_to_resolve_minutes: int


class IncidentMemoryStore:
    """In-memory store for past incident resolutions.
    
    In production, you'd back this with Azure AI Search for semantic retrieval
    or CosmosDB for structured queries.
    """
    
    def __init__(self):
        # Pre-seed with some "past" incidents
        self.memories: list[IncidentMemory] = [
            IncidentMemory(
                service="payment-api",
                title="High latency on payment-api due to memory leak",
                root_cause="PaymentBatchProcessor leaks memory during bulk processing. Pod-3 hit OOM at 4096Mi.",
                resolution="Restarted pod-3. Memory dropped from 3.9GB to 512MB. Latency recovered in 2 minutes. JIRA-4521 tracks permanent fix.",
                timestamp="2026-06-27T03:15:00Z",
                time_to_resolve_minutes=8,
            ),
            IncidentMemory(
                service="payment-api",
                title="Payment API connection pool exhaustion",
                root_cause="Burst of retry storms from order-service exhausted the 50-connection HikariCP pool.",
                resolution="Restarted pod-3 to reset connections. Increased pool size to 75 in next deploy.",
                timestamp="2026-06-25T14:22:00Z",
                time_to_resolve_minutes=12,
            ),
            IncidentMemory(
                service="notification-service",
                title="Email delivery failures - provider rate limiting",
                root_cause="Primary email provider rate-limited us at 1000 emails/hour. Marketing campaign triggered 5000 emails.",
                resolution="Toggled feature flag 'use_backup_email' to route through backup provider. Queue cleared in 15 minutes.",
                timestamp="2026-06-28T19:45:00Z",
                time_to_resolve_minutes=5,
            ),
            IncidentMemory(
                service="user-service",
                title="TLS handshake failures after cert expiry",
                root_cause="Let's Encrypt certificate for user-service.internal expired. cert-manager renewal failed due to DNS challenge timeout.",
                resolution="Manual cert renewal by security team. Auto-remediation NOT possible for cert issues.",
                timestamp="2026-06-05T08:30:00Z",
                time_to_resolve_minutes=35,
            ),
        ]
    
    def search(self, service: str, keywords: str = "") -> list[IncidentMemory]:
        """Find relevant past incidents for a service.
        
        In production, this would use semantic search (embeddings).
        For the workshop, we do simple keyword matching.
        """
        results = []
        keywords_lower = keywords.lower()
        
        for memory in self.memories:
            # Match by service
            if memory.service == service:
                results.append(memory)
            # Also match by keyword in root cause or title
            elif keywords_lower and (
                keywords_lower in memory.root_cause.lower()
                or keywords_lower in memory.title.lower()
            ):
                results.append(memory)
        
        return results
    
    def store(self, memory: IncidentMemory):
        """Store a new incident resolution in memory."""
        self.memories.append(memory)
        print(f"💾 Stored new memory: {memory.title}")


# Initialize the memory store
memory_store = IncidentMemoryStore()
print(f"✅ Memory store initialized with {len(memory_store.memories)} past incidents")

## Memory-Enhanced Triage Agent

Now let's create a Triage Agent that checks memory FIRST — before doing anything else.
If it finds a similar past incident, it can fast-track the response.

### 🎯 YOUR TASK
Create a tool function `search_incident_memory` that wraps the memory store,
then use it in the triage agent.

In [ ]:
from typing import Annotated
from pydantic import Field

# Create a tool function that wraps the memory store
def search_incident_memory(
    service_name: Annotated[str, Field(description="The service to search past incidents for")],
    keywords: Annotated[str, Field(description="Keywords to search for (e.g., 'memory leak', 'rate limit')")] = "",
) -> str:
    """Search the incident memory store for similar past incidents.
    Returns past incidents with their root causes and resolutions."""
    
    results = memory_store.search(service_name, keywords)
    
    if not results:
        return json.dumps({"found": 0, "message": "No similar past incidents found."})
    
    past_incidents = []
    for r in results:
        past_incidents.append({
            "title": r.title,
            "root_cause": r.root_cause,
            "resolution": r.resolution,
            "when": r.timestamp,
            "time_to_resolve_minutes": r.time_to_resolve_minutes,
        })
    
    return json.dumps({"found": len(past_incidents), "past_incidents": past_incidents}, indent=2)


print("✅ search_incident_memory tool created")
# Quick test
print("\nTest search for 'payment-api':")
print(search_incident_memory("payment-api", "memory"))

In [ ]:
async def memory_enhanced_triage():
    """Triage agent that leverages past incident memory."""
    
    with open("data/incidents.json") as f:
        incidents = json.load(f)
    alert = incidents[0]  # payment-api
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # Memory-enhanced triage agent has an EXTRA tool: search_incident_memory
        agent = client.create_agent(
            name="MemoryTriageAgent",
            instructions="""You are a Memory-Enhanced Triage Agent. You have access to a database of past incidents.

When triaging a new alert:
1. FIRST: Use search_incident_memory to check if a similar incident occurred before
2. THEN: Use check_alert_history for recent alert patterns
3. THEN: Use get_runbook for the standard procedure

If you find a matching past incident, reference it explicitly:
- "This matches past incident from [date]: [title]"
- "Previous resolution was: [what worked before]"
- "Expected TTR based on history: [X minutes]"

This helps the Diagnostics and Remediation agents skip unnecessary investigation.""",
            tools=[search_incident_memory, check_alert_history, get_runbook],
        )
        
        result = await agent.run(
            f"New alert:\nTitle: {alert['title']}\nService: {alert['service']}\n"
            f"Severity: {alert['severity']}\nDescription: {alert['description']}\n\n"
            f"Triage this, checking memory first."
        )
        
        print("\n📋 MEMORY-ENHANCED TRIAGE:\n")
        print(result.text)
        return result.text

triage_result = await memory_enhanced_triage()

## Storing New Resolutions

After an incident is resolved, we store the resolution in memory so future incidents benefit.
This creates a **learning loop**.

In [ ]:
# Simulate storing a new resolution after the workflow completes
new_memory = IncidentMemory(
    service="payment-api",
    title="High latency on payment-api - OOM on pod-3 (recurring)",
    root_cause="PaymentBatchProcessor memory leak (3rd occurrence). Pod-3 OOMKilled at 4096Mi limit. Connection pool cascading to order-service.",
    resolution="Restarted payment-api-pod-3. Memory reset to 512MB. Latency recovered. JIRA-4521 priority increased to P1.",
    timestamp=datetime.now().isoformat(),
    time_to_resolve_minutes=6,
)

memory_store.store(new_memory)
print(f"\n📊 Memory store now has {len(memory_store.memories)} incidents")
print(f"   Next time this happens, the agent will resolve even faster.")

## Compare: With vs Without Memory

Notice the difference:

| Without Memory (Step 2) | With Memory (Step 4) |
|---|---|
| Investigates from scratch every time | Immediately recognizes recurring issue |
| Triage takes longer | Fast-tracks known patterns |
| No historical TTR estimate | "Expected resolution: 6-8 minutes" |
| No link to JIRA tickets | "Related: JIRA-4521" |
| Remediation agent may try wrong fix first | Jumps straight to known-good fix |

---

## 🎓 Production Patterns

In a real system, you'd enhance this with:

1. **Azure AI Search** for semantic memory retrieval (not just keyword matching)
2. **Embeddings** to find similar incidents even with different wording
3. **TTL/decay** to phase out outdated memories (infra changes over time)
4. **Confidence scoring** to weight recent resolutions higher
5. **Human feedback loop** — SREs mark if the suggested resolution was actually helpful

## 🏁 Workshop Complete!

### What You Built Today

```
Step 1: Single Agent (Copilot)     → Saw the limitations
Step 2: Specialized Agents + Tools → Task decomposition & tool integration
Step 3: Workflow Orchestration      → Agent coordination with routing
Step 4: Memory Patterns            → Learning system that improves over time
```

### The Journey: Copilot → Orchestrated Agents

| Copilot (Step 1) | Orchestrated Agents (Steps 2-4) |
|---|---|
| Generic advice | Takes real actions |
| No tools | 15 infrastructure tools |
| No memory | Learns from past incidents |
| No verification | Confirms fix worked |
| Can't escalate | Smart routing to humans |
| Same quality forever | Gets better over time |

### Next Steps (After the Workshop)

- **Human-in-the-loop**: Add approval gates before destructive actions
- **Checkpointing**: Persist workflow state for long-running incidents
- **MCP integration**: Connect to real monitoring tools via MCP protocol
- **Observability**: Add OpenTelemetry tracing to monitor agent decisions
- **Evaluation**: Test with red-team scenarios to verify safety guardrails

### Resources

- [Microsoft Agent Framework Docs](https://learn.microsoft.com/en-us/agent-framework/)
- [Agent Framework GitHub](https://github.com/microsoft/agent-framework)
- [Azure AI Foundry](https://ai.azure.com)
- This workshop repo: [github.com/ishasalania/maf-lab](https://github.com/ishasalania/maf-lab)